# 03 Baseline Models (All Datasets)

This notebook trains baseline regressors (Linear Regression, Random Forest, XGBoost) on **all three datasets** (Desharnais, COCOMO-81, China) and saves per-dataset and combined metrics for comparison against CNN+PSO.

In [3]:
from importlib import import_module, util
from pathlib import Path
import random
import sys
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")

drive = None
if util.find_spec("google.colab") is not None:
    drive = import_module("google.colab.drive")
    IN_COLAB = True
else:
    IN_COLAB = False


def resolve_project_root() -> Path:
    if IN_COLAB:
        drive.mount("/content/drive", force_remount=False)
        drive_root = Path("/content/drive/MyDrive")
        candidate_roots = [
            drive_root / "Software-cost-Estimation",
            drive_root / "Colab Notebooks" / "Software-cost-Estimation",
        ]
        for candidate in candidate_roots:
            if (candidate / "src").exists():
                return candidate

        for src_dir in drive_root.rglob("src"):
            if src_dir.is_dir() and src_dir.parent.name == "Software-cost-Estimation":
                return src_dir.parent

        raise FileNotFoundError(
            "Could not find the 'Software-cost-Estimation' project folder in Google Drive. "
            "Upload the full project folder to MyDrive or Colab Notebooks."
        )

    root_dir = Path.cwd()
    if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
        root_dir = root_dir.parent
    return root_dir


root_dir = resolve_project_root()
if not (root_dir / "src").exists():
    raise FileNotFoundError(f"Could not find project root containing 'src' directory. Checked: {root_dir}")

sys.path.insert(0, str(root_dir))
print("Using project root:", root_dir)

from src.cv_pipeline import get_baseline_model_builders
from src.evaluate import evaluate_predictions, save_metrics
from src.feature_engineering import inverse_log_transform, log_transform_target

processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"

# Load all 3 datasets
dataset_files = {
    "desharnais": "desharnais_processed.csv",
    "cocomo81": "cocomo81_processed.csv",
    "china": "china_processed.csv",
}

datasets = {}
for name, fname in dataset_files.items():
    path = processed_dir / fname
    if path.exists():
        datasets[name] = pd.read_csv(path)
        print(f"Loaded {name}: {datasets[name].shape}")
    else:
        print(f"WARNING: {fname} not found, skipping")

print(f"\nLoaded {len(datasets)} datasets")

Using project root: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation
Loaded desharnais: (81, 13)
Loaded cocomo81: (63, 20)
Loaded china: (499, 19)

Loaded 3 datasets


In [4]:
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("xgboost not installed — skipping XGBoost")


def get_baseline_models():
    """Return fresh (untrained) baseline models."""
    models = {
        "linear_regression": LinearRegression(),
        "random_forest": RandomForestRegressor(
            n_estimators=300, random_state=42, n_jobs=-1
        ),
    }
    if HAS_XGBOOST:
        models["xgboost"] = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
        )
    return models

In [5]:
# Train baselines on EACH dataset
all_results = []

for ds_name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name} ({len(df)} samples, {df.shape[1]-1} features)")
    print(f"{'='*60}")

    target_col = df.columns[-1]
    X = df.drop(columns=[target_col])
    y = df[target_col].astype(np.float32)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED
    )

    # Apply the same target transform before every model fit, then invert predictions for metrics.
    y_train_fit = log_transform_target(y_train, use_log_transform=True)

    model_builders = get_baseline_model_builders()
    models = {model_name.lower(): model_builder() for model_name, model_builder in model_builders.items()}

    for model_name, model in models.items():
        model.fit(X_train, y_train_fit)
        preds_fit = np.asarray(model.predict(X_test), dtype=np.float32).ravel()
        preds = inverse_log_transform(preds_fit, use_log_transform=True)
        row = evaluate_predictions(model_name, y_test, preds)
        row["dataset"] = ds_name
        all_results.append(row)
        print(f"  {model_name:20s} MAE={row['mae'].values[0]:10.2f}  RMSE={row['rmse'].values[0]:10.2f}  R²={row['r2'].values[0]:.4f}")

all_baselines = pd.concat(all_results, ignore_index=True)
# Reorder columns
cols = ["dataset", "model", "mae", "rmse", "r2", "mape", "mdmre", "pred25"]
all_baselines = all_baselines[cols].sort_values(["dataset", "rmse"])
all_baselines


Dataset: desharnais (81 samples, 12 features)
  linearregression     MAE=   1392.01  RMSE=   1997.94  R²=0.6871
  randomforest         MAE=   1611.31  RMSE=   2363.72  R²=0.5621
  xgboost              MAE=   1809.61  RMSE=   2548.44  R²=0.4910

Dataset: cocomo81 (63 samples, 19 features)
  linearregression     MAE=    237.60  RMSE=    398.05  R²=0.4873
  randomforest         MAE=    218.10  RMSE=    479.01  R²=0.2575
  xgboost              MAE=    227.56  RMSE=    496.91  R²=0.2010

Dataset: china (499 samples, 18 features)
  linearregression     MAE=   7281.09  RMSE=  53165.28  R²=-93.0377
  randomforest         MAE=    392.21  RMSE=   1636.32  R²=0.9109
  xgboost              MAE=    367.77  RMSE=   1467.32  R²=0.9284


,dataset,model,mae,rmse,r2,mape,mdmre,pred25
8,china,xgboost,367.765243,1467.321352,0.928370,9.590352,0.062659,94.000000
7,china,randomforest,392.208005,1636.323834,0.910919,10.678045,0.074797,98.000000
6,china,linearregression,7281.089682,53165.284246,-93.037716,126.831086,0.483402,18.000000
3,cocomo81,linearregression,237.602681,398.048498,0.487280,279.928784,0.575222,23.076923
4,cocomo81,randomforest,218.101361,479.012902,0.257490,81.578188,0.335540,30.769231
5,cocomo81,xgboost,227.560913,496.914024,0.200956,80.301001,0.344577,38.461538
0,desharnais,linearregression,1392.014064,1997.936250,0.687136,44.289010,0.348296,41.176471
1,desharnais,randomforest,1611.306187,2363.717631,0.562092,62.832478,0.355394,29.411765
2,desharnais,xgboost,1809.605025,2548.441695,0.490973,59.883522,0.420198,17.647059


In [6]:
# Save combined baseline metrics
metrics_path = save_metrics(
    all_baselines,
    file_name="baseline_metrics.csv",
    output_dir=metrics_dir,
)
print("Saved baseline metrics to:", metrics_path)

# Also save per-dataset files for easy lookup
for ds_name in datasets:
    ds_metrics = all_baselines[all_baselines["dataset"] == ds_name]
    save_metrics(ds_metrics, file_name=f"baseline_{ds_name}_metrics.csv", output_dir=metrics_dir)
    print(f"  Saved baseline_{ds_name}_metrics.csv")

Saved baseline metrics to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\baseline_metrics.csv
  Saved baseline_desharnais_metrics.csv
  Saved baseline_cocomo81_metrics.csv
  Saved baseline_china_metrics.csv
